# ACE-Step 1.5 — GPU backend on Kaggle (free) + Cloudflare tunnel

Runs the **ACE-Step engine** (REST API on port 8001) on Kaggle's free GPU and exposes it to your PC
through a public Cloudflare tunnel. Point the web UI's **Base URL** at the printed
`https://*.trycloudflare.com` address.

**Before you start:** right panel → *Settings* → **Accelerator: GPU T4 x2** (or P100), **Internet: On**.

> 16 GB VRAM (T4): use a **2B** model. The XL model will OOM on 16 GB.


## 1. Install uv, clone the engine, and sync dependencies

We must use **uv**, not plain pip: the engine bundles `nano-vllm` as a local package via
`[tool.uv.sources]` and pins CUDA torch from a custom index. `pip install -e .` fails on both.
`uv sync` handles all of it (and auto-fetches a compatible Python 3.11/3.12).


In [ ]:
!pip install -q uv
%cd /kaggle/working
!rm -rf ACE-Step-1.5
!git clone https://github.com/ace-step/ACE-Step-1.5.git
%cd ACE-Step-1.5
!uv sync


## 2. Download cloudflared (free public tunnel, no signup)

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('cloudflared ready')


## 3. Start the API server in the background

Launched via `uv run acestep-api` (the official entry point) on port 8001. On first run it downloads
models (≥4 GB), so we wait ~120s, then show the log. If it is still loading, re-run
`!tail -n 30 /kaggle/working/api.log` until you see it listening on `:8001`.


In [ ]:
import subprocess, os, time

env = os.environ.copy()
env['ACESTEP_API_HOST'] = '0.0.0.0'
env['ACESTEP_API_PORT'] = '8001'

log = open('/kaggle/working/api.log', 'w')
srv = subprocess.Popen(
    ['uv', 'run', 'acestep-api'],
    cwd='/kaggle/working/ACE-Step-1.5',
    stdout=log, stderr=subprocess.STDOUT, env=env,
)
print('Server starting, downloading models (>=4GB). Waiting ~120s...')
time.sleep(120)
print('--- last log lines ---')
!tail -n 30 /kaggle/working/api.log


> If the server needs more time, just re-run: `!tail -n 30 /kaggle/working/api.log`


## 4. Open the public tunnel

This cell **keeps running** and prints a URL like `https://random-words-1234.trycloudflare.com`.
Copy it into the web UI **Base URL** field on your PC (no API key needed for your own engine).


In [ ]:
!cloudflared tunnel --url http://localhost:8001


## Connect the web UI (on your PC)

1. Copy the `https://*.trycloudflare.com` address from cell 4.
2. Run `start_webui.bat` (Windows) and open <http://localhost:5000>.
3. **Base URL** = the tunnel address; leave the API key empty.
4. Click **List models** to verify, then **Generate**.

Keep the Kaggle tab and cell 4 running — closing them kills the tunnel (a restart gives a new URL).
